In [1]:
import pandas as pd
import numpy as np

df=pd.read_pickle("/Users/ismail/Desktop/mlOps/projet/simulated-data-raw/data/2018-04-01.pkl")
df.head()

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO
0,0,2018-04-01 00:00:31,596,3156,57.16,31,0,0,0
1,1,2018-04-01 00:02:10,4961,3412,81.51,130,0,0,0
2,2,2018-04-01 00:07:56,2,1365,146.00,476,0,0,0
3,3,2018-04-01 00:09:29,4128,8737,64.49,569,0,0,0
4,4,2018-04-01 00:10:34,927,9906,50.99,634,0,0,0


In [2]:
df.isna().sum()

TRANSACTION_ID       0
TX_DATETIME          0
CUSTOMER_ID          0
TERMINAL_ID          0
TX_AMOUNT            0
TX_TIME_SECONDS      0
TX_TIME_DAYS         0
TX_FRAUD             0
TX_FRAUD_SCENARIO    0
dtype: int64

In [3]:
df.isnull().sum()

TRANSACTION_ID       0
TX_DATETIME          0
CUSTOMER_ID          0
TERMINAL_ID          0
TX_AMOUNT            0
TX_TIME_SECONDS      0
TX_TIME_DAYS         0
TX_FRAUD             0
TX_FRAUD_SCENARIO    0
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df.dtypes

TRANSACTION_ID                int64
TX_DATETIME          datetime64[ns]
CUSTOMER_ID                  object
TERMINAL_ID                  object
TX_AMOUNT                   float64
TX_TIME_SECONDS              object
TX_TIME_DAYS                 object
TX_FRAUD                      int64
TX_FRAUD_SCENARIO             int64
dtype: object

In [6]:
print(df["TX_FRAUD"].value_counts(normalize=True) * 100)


TX_FRAUD
0    99.968381
1     0.031619
Name: proportion, dtype: float64


In [7]:
print(df["TX_DATETIME"].min(),df["TX_DATETIME"].max())

2018-04-01 00:00:31 2018-04-01 23:59:51


In [8]:
from pathlib import Path
data_dir= Path("/Users/ismail/Desktop/mlOps/projet/simulated-data-raw/data")
files=sorted(data_dir.glob("*.pkl"))
print("Nombre de fichiers :", len(files))

df=pd.concat([pd.read_pickle(file) for file in files],ignore_index=True)
print(df.shape)
print(df["TX_DATETIME"].min(),df["TX_DATETIME"].max())

Nombre de fichiers : 183
(1754155, 9)
2018-04-01 00:00:31 2018-09-30 23:59:57


In [9]:
df["TX_FRAUD"].value_counts()

TX_FRAUD
0    1739474
1      14681
Name: count, dtype: int64

In [10]:
print(df["TX_FRAUD"].value_counts(normalize=True)*100)

TX_FRAUD
0    99.163073
1     0.836927
Name: proportion, dtype: float64


In [11]:
df["TX_DATETIME"] = pd.to_datetime(df["TX_DATETIME"])

df = df.sort_values("TX_DATETIME").reset_index(drop=True)

print("Première transaction :", df["TX_DATETIME"].min())
print("Dernière transaction :", df["TX_DATETIME"].max())

Première transaction : 2018-04-01 00:00:31
Dernière transaction : 2018-09-30 23:59:57


In [12]:
n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

In [13]:
for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(
        name,
        "Taille :", len(data),
        "Fraudes :", data["TX_FRAUD"].sum(),
        "Taux :", round(data["TX_FRAUD"].mean() * 100, 3), "%"
    )

Train Taille : 1227908 Fraudes : 9996 Taux : 0.814 %
Validation Taille : 263123 Fraudes : 2355 Taux : 0.895 %
Test Taille : 263124 Fraudes : 2330 Taux : 0.886 %


In [14]:
# Trier chronologiquement
df = df.sort_values(
    ["CUSTOMER_ID", "TX_DATETIME", "TRANSACTION_ID"]
).copy()

# Moyenne des transactions précédentes uniquement
df["CUSTOMER_AVG_AMOUNT"] = (
    df.groupby("CUSTOMER_ID")["TX_AMOUNT"]
      .transform(lambda x: x.shift(1).expanding().mean())
)


df["AMOUNT_RATIO"] = (
    df["TX_AMOUNT"]/ df["CUSTOMER_AVG_AMOUNT"].replace(0, np.nan)
)

In [15]:
print(df["TX_DATETIME"].dtype)

datetime64[ns]


In [34]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib
from sklearn.metrics import precision_score, recall_score, f1_score
import mlflow.sklearn

for data in [train_df, val_df, test_df]:
    data["TX_HOUR"] = data["TX_DATETIME"].dt.hour
    data["TX_DAY_OF_WEEK"] = data["TX_DATETIME"].dt.day_of_week

features = ["TX_AMOUNT", "TX_HOUR", "TX_DAY_OF_WEEK"]

X_train = train_df[features]
y_train = train_df["TX_FRAUD"]

X_val = val_df[features]
y_val = val_df["TX_FRAUD"]

import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("fraud-detection-unbalanced")


with mlflow.start_run(run_name="logistic-regression-balanced"):

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
        class_weight=None,
        max_iter=1000
    ))
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_val)

    print(classification_report(y_val, predictions, zero_division=0))

    mlflow.log_metric("precision", precision_score(y_val, predictions))
    mlflow.log_metric("recall", recall_score(y_val, predictions))
    mlflow.log_metric("f1_score", f1_score(y_val, predictions))

    joblib.dump(pipeline, "fraud_model.joblib")

    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("class_weight", "None")
    mlflow.log_param("max_iter", 1000)

    mlflow.sklearn.log_model(
        sk_model=pipeline,
        name="fraud_model"
    )


print("Experiment successfully recorded in MLflow!")


              precision    recall  f1-score   support

           0       0.99      1.00      1.00    260768
           1       1.00      0.10      0.18      2355

    accuracy                           0.99    263123
   macro avg       1.00      0.55      0.59    263123
weighted avg       0.99      0.99      0.99    263123

🏃 View run logistic-regression-balanced at: http://127.0.0.1:5000/#/experiments/2/runs/0886616467494be4bda8f4004ef5918b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Experiment successfully recorded in MLflow!


In [33]:
mlflow.end_run()